# Pure Python Implementation (Baseline)

This is the starting point – a clean, straightforward implementation in pure Python using lists of lists. No optimizations, no NumPy, no Numba, nothing fancy.

I split it into small functions to keep it readable. For the 1000×1000 grid it would take ages (hours), so I'm running it on a smaller 80×80 grid (only in this example) with 10,000 generations to get a reasonable baseline time that we can compare against later versions.

In [18]:
import random
import time

start = time.perf_counter()


SIZE = 80
GENERATIONS = 10000

def create_random_grid(size):
    grid = []
    for i in range(size):
        row = []
        for j in range(size):
            row.append(random.randint(0, 1))
        grid.append(row)
    return grid

def count_neighbors(grid, i, j, size):
    neighbors = 0
    for di in [-1, 0, 1]:
        for dj in [-1, 0, 1]:
            if di == 0 and dj == 0:
                continue
            ni = (i + di) % size
            nj = (j + dj) % size
            neighbors += grid[ni][nj]
    return neighbors

def next_generation(grid, size):
    new_grid = [row[:] for row in grid]
    for i in range(size):
        for j in range(size):
            neighbors = count_neighbors(grid, i, j, size)
            if grid[i][j] == 1:
                if neighbors < 2 or neighbors > 3:
                    new_grid[i][j] = 0
            else:
                if neighbors == 3:
                    new_grid[i][j] = 1
    return new_grid

def run_game(size, generations):
    grid = create_random_grid(size)
    for gen in range(generations):
        grid = next_generation(grid, size)
    return grid

cell_matrix = run_game(SIZE, GENERATIONS)

end = time.perf_counter()
print(f"Execution time: {end - start:.6f} s")

Execution time: 50.602483 s


### Numba CPU (single-threaded @njit)

Next step – let's see what Numba can do on the CPU without parallelization.  
I decorated all functions with `@njit` and switched to NumPy arrays (Numba loves them).  
Random grid generation is still inside `@njit` (using `random.randint`, which works fine here).

This should already be much faster than pure Python, even on a single core.  

In [19]:
import random
import time
import numpy as np
from numba import njit, prange

start = time.perf_counter()

SIZE = 1000
GENERATIONS = 10000

@njit
def create_random_grid(size):
    grid = np.zeros((size, size), dtype=np.int8)
    for i in range(size):
        for j in range(size):
            grid[i, j] = random.randint(0, 1)
    return grid

@njit
def count_neighbors(grid, i, j, size):
    neighbors = 0
    for di in [-1, 0, 1]:
        for dj in [-1, 0, 1]:
            if di == 0 and dj == 0:
                continue
            ni = (i + di) % size
            nj = (j + dj) % size
            neighbors += grid[ni, nj]
    return neighbors

@njit
def next_generation(grid, size):
    new_grid = grid.copy()
    for i in range(size):
        for j in range(size):
            neighbors = count_neighbors(grid, i, j, size)
            if grid[i, j] == 1:
                if neighbors < 2 or neighbors > 3:
                    new_grid[i, j] = 0
            else:
                if neighbors == 3:
                    new_grid[i, j] = 1
    return new_grid

@njit
def run_game(size, generations):
    grid = create_random_grid(size)
    for _ in range(generations):
        grid = next_generation(grid, size)
    return grid

cell_matrix = run_game(SIZE, GENERATIONS)

end = time.perf_counter()
print(f"Execution time: {end - start:.6f} s")

Execution time: 243.755216 s


### Numba CPU with Multi-Threading (prange)

Now we go parallel on the CPU.  
I merged everything into one big function and used `@njit(parallel=True)` with `prange` on the outer loop – each row gets its own thread.

Random initialization uses `np.random.rand()` (fully supported in Numba).  
This should scale nicely with the number of cores.

In [20]:
import time
import numpy as np
from numba import njit, prange

start = time.perf_counter()

SIZE = 1000
GENERATIONS = 10000

@njit(parallel=True, cache=True)
def game_of_life(size, generations):
    grid = np.zeros((size, size), dtype=np.int8)
    
    for i in range(size):
        for j in range(size):
            if np.random.rand() < 0.5:
                grid[i, j] = 1

    for _ in range(generations):
        new_grid = np.empty_like(grid)

        for i in prange(size):
            for j in range(size):
                neighbors = 0
                for di in [-1, 0, 1]:
                    for dj in [-1, 0, 1]:
                        if di == 0 and dj == 0:
                            continue
                        ni = (i + di) % size
                        nj = (j + dj) % size
                        neighbors += grid[ni, nj]

                if grid[i, j] == 1:
                    if neighbors < 2 or neighbors > 3:
                        new_grid[i, j] = 0
                    else:
                        new_grid[i, j] = 1
                else:
                    if neighbors == 3:
                        new_grid[i, j] = 1
                    else:
                        new_grid[i, j] = 0

        grid = new_grid

    return grid

result = game_of_life(SIZE, GENERATIONS)

end = time.perf_counter()
print(f"Execution time: {end - start:.3f} sec")

Execution time: 46.319 sec


### Numba CUDA – Basic GPU Implementation

This implementation uses Numba CUDA with a simple "one thread per cell" approach.  
The grid is flattened to 1D for easier indexing, and all memory accesses are from global memory (no shared memory optimization).

It provides substantial speedup over CPU versions, especially on larger grids.  

In [21]:
import time
import numpy as np
from numba import cuda

start = time.perf_counter()

SIZE = 1000
GENERATIONS = 10000
THREADS_PER_BLOCK = 256
BLOCKS_PER_GRID = (SIZE * SIZE + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK

@cuda.jit
def game_of_life_kernel(grid_in, grid_out, size):
    idx = cuda.grid(1)
    if idx >= size * size:
        return
    
    i = idx // size
    j = idx % size
    
    neighbors = 0
    for di in [-1, 0, 1]:
        for dj in [-1, 0, 1]:
            if di == 0 and dj == 0:
                continue
            ni = (i + di) % size
            nj = (j + dj) % size
            neighbors += grid_in[ni * size + nj]
    
    alive = grid_in[idx]
    if alive == 1:
        if neighbors < 2 or neighbors > 3:
            grid_out[idx] = 0
        else:
            grid_out[idx] = 1
    else:
        if neighbors == 3:
            grid_out[idx] = 1
        else:
            grid_out[idx] = 0

grid = np.random.randint(0, 2, size=(SIZE, SIZE), dtype=np.int8).ravel()

d_grid_in = cuda.to_device(grid)
d_grid_out = cuda.to_device(np.zeros_like(grid))

for _ in range(GENERATIONS):
    game_of_life_kernel[BLOCKS_PER_GRID, THREADS_PER_BLOCK](d_grid_in, d_grid_out, SIZE)
    d_grid_in, d_grid_out = d_grid_out, d_grid_in

result = d_grid_in.copy_to_host().reshape((SIZE, SIZE))

end = time.perf_counter()
print(f"Execution time: {end - start:.3f} sec")

Execution time: 4.478 sec


### Taichi – GPU Implementation

This section uses Taichi, a Python-embedded domain-specific language for high-performance computing on GPU.

The code is concise and readable, similar to NumPy-style syntax, while running entirely on the GPU.  
Neighbor counting is performed with simple nested loops (no static loops to avoid continue issues).

Taichi typically provides the best performance-to-code-simplicity ratio among GPU options in Python.

In [22]:
import time
import numpy as np
import taichi as ti

ti.init(arch=ti.gpu) 

start = time.perf_counter()

SIZE = 1000       
GENERATIONS = 10000

grid = ti.field(dtype=ti.i32, shape=(SIZE, SIZE))
new_grid = ti.field(dtype=ti.i32, shape=(SIZE, SIZE))

@ti.kernel
def step():
    for i, j in grid:
        neighbors = 0
        for di in range(-1, 2):
            for dj in range(-1, 2):
                if di == 0 and dj == 0:
                    continue
                ni = (i + di) % SIZE
                nj = (j + dj) % SIZE
                neighbors += grid[ni, nj]

        alive = grid[i, j]
        if alive == 1:
            new_grid[i, j] = 1 if 2 <= neighbors <= 3 else 0
        else:
            new_grid[i, j] = 1 if neighbors == 3 else 0

    for i, j in grid:
        grid[i, j] = new_grid[i, j]

grid.from_numpy(np.random.randint(0, 2, (SIZE, SIZE), dtype=np.int32))

for _ in range(GENERATIONS):
    step()

end = time.perf_counter()
print(f"Taichi GPU time: {end - start:.3f} seconds")

[Taichi] Starting on arch=cuda
Taichi GPU time: 1.094 seconds


### CuPy + SciPy Convolution (Hybrid CPU/GPU)

This implementation combines CuPy for GPU memory management with SciPy's `convolve2d` for neighbor counting on the CPU.

Data is transferred between GPU (CuPy array) and CPU (NumPy array) each generation.  
Convolution is performed on CPU using a 3×3 kernel, followed by rule application with `np.where`.

While not purely GPU-based (due to data transfers), it is simple and leverages optimized SciPy routines.  
Performance is reasonable, but limited by CPU-GPU transfer overhead on larger grids.

In [23]:
import time
import numpy as np
import cupy as cp
from scipy.signal import convolve2d

start = time.perf_counter()

SIZE = 1000
GENERATIONS = 10000

kernel = np.array([[1,1,1],
                   [1,0,1],
                   [1,1,1]], dtype=np.int32)

grid_cp = cp.random.randint(0, 2, (SIZE, SIZE), dtype=cp.int8)

for _ in range(GENERATIONS):
    grid_np = cp.asnumpy(grid_cp)
    
    neighbors = convolve2d(grid_np, kernel, mode='same', boundary='wrap')
    
    new_grid_np = np.where((grid_np == 1) & ((neighbors == 2) | (neighbors == 3)), 1,
                           np.where((grid_np == 0) & (neighbors == 3), 1, 0))
    
    grid_cp = cp.asarray(new_grid_np)

end = time.perf_counter()
print(f"CuPy + SciPy convolution time: {end - start:.3f} seconds")

CuPy + SciPy convolution time: 335.004 seconds
